In [0]:
from pyspark.sql import functions as F

sk = spark.table("jobmarket.silver.silver_posting_skills")
sp = spark.table("jobmarket.silver.silver_job_postings")

# disclosed salaries only (Decision 2), soft skills excluded (Decision 4)
salaried = (sp
    .filter(F.col("salary_is_estimated") == False)
    .filter(F.col("salary_min").isNotNull())
    .withColumn("salary_mid",
        F.when(F.col("salary_max").isNotNull(),
               (F.col("salary_min") + F.col("salary_max")) / 2)
         .otherwise(F.col("salary_min")))
    .select("posting_id", "state", "salary_mid"))

skill_salary = (sk
    .filter(F.col("skill_group") != "soft")
    .join(salaried, "posting_id")
    .groupBy("skill", "skill_group", "state")
    .agg(
        F.round(F.expr("percentile_approx(salary_mid, 0.25)")).alias("p25_salary"),
        F.round(F.expr("percentile_approx(salary_mid, 0.5)")).alias("median_salary"),
        F.round(F.expr("percentile_approx(salary_mid, 0.75)")).alias("p75_salary"),
        F.countDistinct("posting_id").alias("sample_size")))

(skill_salary.write.format("delta").mode("overwrite")
    .saveAsTable("jobmarket.gold.gold_skill_salary"))

gs = spark.table("jobmarket.gold.gold_skill_salary")
print("Rows total:", gs.count())
print("Rows passing the floor:", gs.filter("sample_size >= 30").count())

# sanity: national picture, floor applied — do the numbers make sense?
gs.filter("sample_size >= 30") \
  .orderBy(F.desc("median_salary")).limit(10).display()

In [0]:
gs.filter("sample_size >= 30").agg(F.min("sample_size")).display()   # must be >= 30